In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
spark=SparkSession.builder.appName("wine_MultiClass").getOrCreate()

In [5]:
wine_df=spark.read.csv("wine.csv", header=True, inferSchema=True)

In [6]:
wine_df.show()

+----+-------+----------+----+----+---+-------+----------+--------------------+-------+---------+----+----+-------+
|Wine|Alcohol|Malic.acid| Ash| Acl| Mg|Phenols|Flavanoids|Nonflavanoid.phenols|Proanth|Color.int| Hue|  OD|Proline|
+----+-------+----------+----+----+---+-------+----------+--------------------+-------+---------+----+----+-------+
|   1|  14.23|      1.71|2.43|15.6|127|    2.8|      3.06|                0.28|   2.29|     5.64|1.04|3.92|   1065|
|   1|   13.2|      1.78|2.14|11.2|100|   2.65|      2.76|                0.26|   1.28|     4.38|1.05| 3.4|   1050|
|   1|  13.16|      2.36|2.67|18.6|101|    2.8|      3.24|                 0.3|   2.81|     5.68|1.03|3.17|   1185|
|   1|  14.37|      1.95| 2.5|16.8|113|   3.85|      3.49|                0.24|   2.18|      7.8|0.86|3.45|   1480|
|   1|  13.24|      2.59|2.87|21.0|118|    2.8|      2.69|                0.39|   1.82|     4.32|1.04|2.93|    735|
|   1|   14.2|      1.76|2.45|15.2|112|   3.27|      3.39|              

In [11]:
wine_df=wine_df.withColumnRenamed("Malic.acid", "Malic_acid",).withColumnRenamed("Nonflavanoid.phenols","Nonflavanoid_phenols").withColumnRenamed("Color.int","Color_int")

In [12]:
# Assemble features
assembler = VectorAssembler(inputCols=["Alcohol","Malic_acid","Ash","Acl", "Mg","Phenols","Flavanoids","Nonflavanoid_phenols","Proanth","Color_int","Hue","OD", "Proline"], outputCol="features")
wine_df = assembler.transform(wine_df)

In [13]:
#Split the data
train_df,test_df=wine_df.randomSplit([0.8,0.2],seed=42)

In [16]:
#Create and train the model
rf=RandomForestClassifier(labelCol="Wine", featuresCol="features", numTrees=10)
model=rf.fit(train_df)

In [17]:
#Evaluate the model
predictions=model.transform(test_df)
evaluator=MulticlassClassificationEvaluator(metricName="accuracy", labelCol="Wine", predictionCol="prediction")
accuracy=evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

Accuracy: 1.0
